# Exercise 03 — batch producer & partition keys

**Goal:** send many events at once and **see, in your own output**, the consequence of choosing a key (or not).

Why care? In real systems, ordering matters. Anna's order goes through `placed` → `accepted` → `picked_up` → `delivered`. If `delivered` arrived before `picked_up`, the app would show her pizza as delivered and then as on its way again. Kafka guarantees order **per partition**, so you must put related events on the same partition by giving them the same key — here the order id.

The setup cell below prepares four riders and three orders and a helper `make_event(topic)` that returns either a GPS ping of a random rider (key = rider id) or the next status of a random order (key = order id).

In [ ]:
from confluent_kafka import Producer
from collections import defaultdict, Counter
import json, time, random

BROKER = 'redpanda:29092'

producer = Producer({
    'bootstrap.servers': BROKER,
    'client.id':         'batch-producer',
})

riders = ['rider_3', 'rider_5', 'rider_6', 'rider_7']
orders = {                       # order id -> (customer, restaurant, total in CHF)
    'order-4711': ('Anna',  'Pizzeria Roma',   32.50),
    'order-4712': ('Ben',   'Sushi Bar Kyoto', 47.80),
    'order-4713': ('Clara', 'Burger House',    21.90),
}
STATUSES   = ['placed', 'accepted', 'picked_up', 'delivered']
order_step = {order_id: 0 for order_id in orders}   # how far each order has progressed
topics     = ['rider-locations', 'orders']


def make_event(topic):
    """Return (key, event) — a GPS ping of a random rider or the next status of a random order."""
    if topic == 'rider-locations':
        rider = random.choice(riders)
        return rider, {
            'rider_id':  rider,
            'lat':       round(47.0502 + random.uniform(-0.01, 0.01), 5),
            'lon':       round(8.3093 + random.uniform(-0.01, 0.01), 5),
            'speed_kmh': round(random.uniform(0.0, 35.0), 1),
            'timestamp': time.time(),
        }
    order_id = random.choice(list(orders))
    customer, restaurant, total = orders[order_id]
    status = STATUSES[min(order_step[order_id], len(STATUSES) - 1)]
    order_step[order_id] += 1
    return order_id, {
        'order_id':   order_id,
        'customer':   customer,
        'restaurant': restaurant,
        'total_chf':  total,
        'status':     status,
        'timestamp':  time.time(),
    }

## Step 1 — send 15 random events

Each event picks a random topic, and within it a random rider or order. We capture the broker's *delivery report* (which contains the assigned partition) so we can analyse the mapping after the fact.

**Watch:** the same `key` always returns the same `partition`, regardless of how often you re-run.

In [ ]:
results = []

def delivery_report(err, msg):
    if err:
        print(f'FAILED: {err}')
    else:
        results.append({
            'topic':     msg.topic(),
            'partition': msg.partition(),
            'key':       msg.key().decode(),
        })
        print(f'  -> {msg.topic()} [P{msg.partition()}] '
              f'key={msg.key().decode()} offset={msg.offset()}')

for _ in range(15):
    topic      = random.choice(topics)
    key, event = make_event(topic)
    producer.produce(topic, key=key.encode(), value=json.dumps(event).encode(),
                     callback=delivery_report)

producer.flush()
print('15 events sent.')

## Step 2 — analyse the key→partition mapping

We group the deliveries by `(topic, key)` and check whether each pair always landed on the same partition. If `consistent? = yes`, the key-hashing is doing its job.

In [ ]:
key_partitions = defaultdict(set)
for r in results:
    key_partitions[(r['topic'], r['key'])].add(r['partition'])

print(f'{"Topic":<15} | {"Key":<10} | Partition(s) | Consistent?')
print('-' * 60)
for (t, k), parts in sorted(key_partitions.items()):
    plist = ', '.join(str(p) for p in sorted(parts))
    flag  = 'yes' if len(parts) == 1 else 'NO!'
    print(f'{t:<15} | {k:<10} | {plist:<12} | {flag}')

## Task A — events without a key

**Send 6 pings to `rider-locations` with `key=None`.** Put a `rider_id` into the JSON value as usual — the point is that Kafka only looks at the *key* for routing, never inside the value. Where do they land?

**Expected:** the producer falls back to a *sticky* partitioner — it fills one partition for a while, then switches. So you'll typically see all 6 on the same partition, *but* there's no guarantee — if you re-run, the partition can change.

**Consequence:** keyless events have **no ordering guarantee** between each other. If order matters, give them a key.

In [ ]:
no_key_partitions = []

def no_key_cb(err, msg):
    if not err:
        no_key_partitions.append(msg.partition())
        print(f'  -> P{msg.partition()} offset={msg.offset()}')

# TODO: loop 6 times, produce a small JSON ping to 'rider-locations'
#       with key=None and callback=no_key_cb

producer.flush()
print(f'Distribution: {dict(Counter(no_key_partitions))}')

## Task B — confirm in the Console

1. Open Redpanda Console → topic `rider-locations` → *Messages*.
2. Use the *Partition* filter — first **0** only, then **1**, then **2**.
3. Compare the keys you see with what your Python output reported above. They should match exactly. The keyless pings show up with an empty key.

## What you learned

- Choosing a key gives **deterministic, ordered** routing inside a topic.
- No key → load is spread but **order is lost** between events.
- The same key always lands on the same partition, **as long as the partition count doesn't change**. Re-creating a topic with a different partition count will reshuffle everything.